<a href="https://colab.research.google.com/github/lgplorea/EjerciciosPython/blob/main/AlgoritmoRegresionAdopcion_Muestra200.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Calculamos la media de los días de estancia:

In [9]:
# ==========================================================
# PROGRAMA: Calcular la media de los días de estancia
# ==========================================================

# La "media" es simplemente: sumar todos los días de estancia
# y dividir esa suma entre el número total de animales.
# ==========================================================

import csv  # herramienta de Python que sirve para leer archivos .csv

# 1. Indicamos el nombre del archivo que queremos leer.
nombre_archivo = "bbdd_muestra_200.csv"

# 2. Creamos una lista vacía donde guardaremos todos los
#    valores de "dias_estancia" que vayamos encontrando.
lista_dias_estancia = []

# 3. Abrimos el archivo CSV para poder leerlo.
with open(nombre_archivo, mode="r", encoding="utf-8") as archivo:

    # csv.DictReader nos permite leer cada fila como si fuera
    # un diccionario, es decir, podemos pedir el valor de una
    # columna usando su nombre (por ejemplo: fila["dias_estancia"])
    lector = csv.DictReader(archivo)

    # 4. Recorremos el archivo fila por fila (cada fila es un animal)
    for fila in lector:
        # Tomamos el valor de la columna "dias_estancia" de esa fila
        valor = fila["dias_estancia"]

        # Lo convertimos a número (int = número entero),
        # porque al leerlo del CSV viene como texto
        dias = int(valor)

        # Lo añadimos a nuestra lista
        lista_dias_estancia.append(dias)

# 5. Calculamos la media:
#    - sum(lista) suma todos los números de la lista
#    - len(lista) cuenta cuántos números tiene la lista
cantidad_animales = len(lista_dias_estancia)

media = sum(lista_dias_estancia) / cantidad_animales

# 6. Mostramos los resultados en pantalla
print("Número total de animales:", cantidad_animales)
print("Media de días de estancia:", round(media, 2))

Número total de animales: 200
Media de días de estancia: 63.09


Será una regresión lineal múltiple usando varias variables como predictoras de dias_estancia.

¿Qué variables vamos a usar para predecir los días de estancia?

Usar: Intake Type, Intake Condition, Animal Type, Sex upon Intake, Age upon Intake (convertida a número de días/meses).

Excluir: Animal ID, Name, DateTime, MonthYear, Found Location, Breed, Color, id_estancia, fechas (son identificadores o texto libre, no aptos para una regresión sencilla), Outcome Type, Outcome Subtype

EDAD: Tiene formatos variados (días, semanas, meses, años) que convertiré todos a edad en días para que sea un número comparable.

In [10]:
# ==========================================================
# PROGRAMA: Regresión lineal para predecir los días de estancia
# (usando solo los datos disponibles en el momento del ingreso)
# ==========================================================
# Variables que usamos:
#   - Intake Type        (tipo de ingreso)
#   - Intake Condition    (condición de ingreso)
#   - Animal Type         (tipo de animal)
#   - Sex upon Intake     (sexo al ingresar)
#   - Age upon Intake     (edad al ingresar)


import pandas as pd
import math
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Leemos el archivo CSV y lo guardamos en una tabla (DataFrame)
datos = pd.read_csv("bbdd_muestra_200.csv")

# 2. Elegimos las columnas que queremos usar para predecir los días de estancia.
columnas_predictoras = [
    "Intake Type",
    "Intake Condition",
    "Animal Type",
    "Sex upon Intake",
    "Age upon Intake",
]

# 3. La columna "Age upon Intake" viene como texto (ej: "2 years",
#    "3 months"). La convertimos a un número de días para que el
#    modelo pueda usarla.
def convertir_edad_a_dias(texto_edad):
    numero, unidad = texto_edad.split()   # separa "2 years" en "2" y "years"
    numero = int(numero)

    if "day" in unidad:
        return numero
    elif "week" in unidad:
        return numero * 7
    elif "month" in unidad:
        return numero * 30
    elif "year" in unidad:
        return numero * 365
    else:
        return None

datos["Age upon Intake"] = datos["Age upon Intake"].apply(convertir_edad_a_dias)

# 4. Las demás columnas predictoras son texto (ej: "Dog", "Cat",
#    "Stray"...). Los modelos de regresión solo entienden números,
#    así que convertimos cada categoría de texto en columnas de
#    0 y 1 (esto se llama "one-hot encoding").
columnas_texto = [
    "Intake Type",
    "Intake Condition",
    "Animal Type",
    "Sex upon Intake",
]

datos_preparados = pd.get_dummies(datos[columnas_predictoras], columns=columnas_texto)

# 5. Definimos:
#    X = las variables que usamos para predecir (todas las preparadas)
#    y = lo que queremos predecir (los días de estancia)
X = datos_preparados
y = datos["dias_estancia"]

# 6. Creamos el modelo de regresión lineal y lo entrenamos con
#    nuestros datos (esto es lo que se llama "ajustar" el modelo)
modelo = LinearRegression()
modelo.fit(X, y)

# 7. Calculamos qué tan bien predice el modelo, con el "R²"
#    (R² va de 0 a 1: cuanto más cerca de 1, mejor explica el
#    modelo los días de estancia con estas variables)
r2 = modelo.score(X, y)

# 8. Calculamos el error promedio del modelo (MAE = Mean Absolute
#    Error, o "Error Absoluto Medio"). El resultado nos dice, en
#    promedio, cuántos días se equivoca el modelo en cada predicción.
predicciones = modelo.predict(X)
mae = mean_absolute_error(y, predicciones)

# 9. Calculamos el MSE (Mean Squared Error, o "Error Cuadrático
#    Medio"). Es parecido al MAE, pero aquí los errores se elevan
#    al cuadrado antes de promediarlos. Esto hace que los errores
#    grandes "pesen" mucho más, por lo que el MSE es útil para
#    detectar si el modelo falla fuerte en algunos casos puntuales.
mse = mean_squared_error(y, predicciones)

# 10. Calculamos el RMSE (Root Mean Squared Error, o "Raíz del
#     Error Cuadrático Medio"). Es la raíz cuadrada del MSE. Esto
#     "deshace" el efecto de elevar al cuadrado y devuelve el
#     error a la misma unidad que los días de estancia, por lo
#     que es más fácil de interpretar que el MSE.
rmse = math.sqrt(mse)

# 11. Mostramos los resultados
print("===== RESULTADOS DE LA REGRESIÓN LINEAL =====")
print("R² del modelo:", round(r2, 3))
print()
print("Esto significa que el modelo explica el",
      round(r2 * 100, 1), "% de la variación en los días de estancia.")
print()
print("MAE (Error Absoluto Medio):", round(mae, 2), "días")
print("(en promedio, la predicción del modelo se equivoca por esta cantidad de días)")
print()
print("MSE (Error Cuadrático Medio):", round(mse, 2))
print("(es el error promedio, pero elevado al cuadrado; penaliza más los errores grandes)")
print()
print("RMSE (Raíz del Error Cuadrático Medio):", round(rmse, 2), "días")
print("(la diferencia típica entre el valor predicho y el real, ya en días)")
print()
print("Las 5 variables con mayor influencia (positiva o negativa):")

# Creamos una tabla con el nombre de cada variable y su coeficiente
# (el coeficiente indica cuánto aumenta o disminuye la predicción
# de días de estancia por cada variable)
coeficientes = pd.Series(modelo.coef_, index=X.columns)
coeficientes_ordenados = coeficientes.abs().sort_values(ascending=False).head(5)

for variable in coeficientes_ordenados.index:
    print(f"  - {variable}: {round(coeficientes[variable], 2)}")

===== RESULTADOS DE LA REGRESIÓN LINEAL =====
R² del modelo: 0.212

Esto significa que el modelo explica el 21.2 % de la variación en los días de estancia.

MAE (Error Absoluto Medio): 12.54 días
(en promedio, la predicción del modelo se equivoca por esta cantidad de días)

MSE (Error Cuadrático Medio): 265.72
(es el error promedio, pero elevado al cuadrado; penaliza más los errores grandes)

RMSE (Raíz del Error Cuadrático Medio): 16.3 días
(la diferencia típica entre el valor predicho y el real, ya en días)

Las 5 variables con mayor influencia (positiva o negativa):
  - Intake Condition_Sick: 12.13
  - Intake Condition_Neonatal: -10.74
  - Intake Type_Public Assist: -10.47
  - Animal Type_Other: -9.68
  - Intake Condition_Injured: 9.51


Creamos inputs para predecir los días de estancia de casos nuevos:

In [11]:
# ==========================================================
# FORMULARIO: Predecir los días de estancia de un caso nuevo
# ==========================================================

# 1. Opciones válidas para cada pregunta
# Mostrar estas listas evita que el usuario escriba algo mal .
# El usuario solo elige un número, y el programa traduce ese número.

opciones_intake_type = ["Owner Surrender", "Public Assist", "Stray", "Wildlife"]
opciones_intake_condition = ["Injured", "Neonatal", "Normal", "Nursing", "Sick"]
opciones_animal_type = ["Cat", "Dog", "Other"]
opciones_sex = ["Intact Female", "Intact Male", "Neutered Male", "Spayed Female", "Unknown"]


def preguntar_opcion(pregunta, opciones):

    print("\n" + pregunta)
    for i, opcion in enumerate(opciones, start=1):
        print(f"  {i}. {opcion}")

    while True:
        respuesta = input("Elige el número de tu opción: ").strip()
        if respuesta.isdigit() and 1 <= int(respuesta) <= len(opciones):
            return opciones[int(respuesta) - 1]
        print("Opción no válida. Por favor, introduce solo el número de la lista.")


def preguntar_edad():

    print("\n¿Cuál es la edad del animal?")

    while True:
        numero = input("Introduce un número (ej: 2): ").strip()
        if numero.isdigit():
            numero = int(numero)
            break
        print("Eso no es un número válido. Inténtalo de nuevo.")

    unidades = ["days", "weeks", "months", "years"]
    unidad = preguntar_opcion("¿En qué unidad (la edad)? (elige una)", unidades)

    if unidad == "days":
        return numero
    elif unidad == "weeks":
        return numero * 7
    elif unidad == "months":
        return numero * 30
    elif unidad == "years":
        return numero * 365


# 2. Hacemos las preguntas al usuario

print("\n===== INTRODUCE LOS DATOS DEL NUEVO ANIMAL =====")

respuesta_intake_type = preguntar_opcion("¿Cuál es el tipo de ingreso?", opciones_intake_type)
respuesta_intake_condition = preguntar_opcion("¿Cuál es la condición de ingreso?", opciones_intake_condition)
respuesta_animal_type = preguntar_opcion("¿Qué tipo de animal es?", opciones_animal_type)
respuesta_sex = preguntar_opcion("¿Cuál es el sexo del animal?", opciones_sex)
respuesta_edad_en_dias = preguntar_edad()


# 3. Construimos el caso nuevo en el mismo formato que el modelo espera ----------

# X.columns contiene el nombre y el orden exacto de las columnas
# que el modelo aprendió a usar. Empezamos poniendo todo a 0...
columnas_del_modelo = X.columns
caso_nuevo = pd.DataFrame([[0] * len(columnas_del_modelo)], columns=columnas_del_modelo)

# ...y vamos activando (poniendo a 1) o rellenando solo las columnas
# que corresponden a las respuestas del usuario.
caso_nuevo["Age upon Intake"] = respuesta_edad_en_dias
caso_nuevo[f"Intake Type_{respuesta_intake_type}"] = 1
caso_nuevo[f"Intake Condition_{respuesta_intake_condition}"] = 1
caso_nuevo[f"Animal Type_{respuesta_animal_type}"] = 1
caso_nuevo[f"Sex upon Intake_{respuesta_sex}"] = 1


# ---------- 4. Hacemos la predicción ----------

prediccion_dias = modelo.predict(caso_nuevo)[0]

print("\n===== RESULTADO =====")
print(f"Predicción de días de estancia: {round(prediccion_dias, 1)} días")
print(f"(margen de error promedio del modelo: ± {round(rmse, 1)} días, según el RMSE calculado antes)")


===== INTRODUCE LOS DATOS DEL NUEVO ANIMAL =====

¿Cuál es el tipo de ingreso?
  1. Owner Surrender
  2. Public Assist
  3. Stray
  4. Wildlife
Elige el número de tu opción: 2

¿Cuál es la condición de ingreso?
  1. Injured
  2. Neonatal
  3. Normal
  4. Nursing
  5. Sick
Elige el número de tu opción: 3

¿Qué tipo de animal es?
  1. Cat
  2. Dog
  3. Other
Elige el número de tu opción: 2

¿Cuál es el sexo del animal?
  1. Intact Female
  2. Intact Male
  3. Neutered Male
  4. Spayed Female
  5. Unknown
Elige el número de tu opción: 5

¿Cuál es la edad del animal?
Introduce un número (ej: 2): 2

¿En qué unidad (la edad)? (elige una)
  1. days
  2. weeks
  3. months
  4. years
Elige el número de tu opción: 3

===== RESULTADO =====
Predicción de días de estancia: 44.1 días
(margen de error promedio del modelo: ± 16.3 días, según el RMSE calculado antes)
